# ⬡ J.A.R.V.I.S. · VoxCPM2 Hindi / Hinglish TTS Engine · v4
### *Readlyte Production · OpenBMB VoxCPM2 · 2B Params · 48 kHz · 30 Languages*

---
> **Run all cells top-to-bottom (Runtime → Run all).**  
> Set Runtime Type to **T4 GPU** before starting.

### What's new in v4
| Fix | Detail |
|-----|--------|
| **Style cue parsing fixed** | `(instruction)text` paragraphs now correctly pass the instruction to the model — it is no longer spoken aloud. |
| **STYLE_PREFIX removed** | Was prepended before `(instruction)`, breaking VoxCPM2's own format. Removed entirely. Voice is controlled 100% by the in-text parentheticals. |
| **Style cue regex relaxed** | Minimum length reduced from 5 to 3 chars; trailing period/comma in cue no longer disqualifies it. |
| **Normalization checkbox** | `ENABLE_NORMALIZATION` exposed as a Colab form checkbox in Cell 3. |
| **Boilerplate trimmed** | Dashboard HTML helpers consolidated; notebook size reduced ~40%. |

| Cell | Purpose |
|------|---------|
| 1 | GPU / Environment Diagnostics |
| 2 | Install Dependencies |
| 3 | Upload Text + Preprocessing |
| 4 | Reference Audio (optional voice clone) |
| 5 | TTS Config |
| 6 | Generate & Stitch |
| 7 | Playback & Download |

In [ ]:
# ================================================================
# CELL 1 — SYSTEM DIAGNOSTICS
# ================================================================
import subprocess, sys, os, platform
from IPython.display import display, HTML

# ── Shared dashboard helpers (available to ALL cells) ────────────

def _esc(s):
    return str(s).replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')

def _panel(title, rows, accent='#e74c3c', note='', width='700px'):
    inner = ''
    for row in rows:
        if row is None:
            inner += '<tr><td colspan="3"><div style="border-top:1px solid #1e1e1e;margin:5px 0"></div></td></tr>'
            continue
        if isinstance(row, str):
            inner += (f'<tr><td colspan="3" style="color:{accent};font-size:10px;letter-spacing:2px;'
                      f'padding:9px 0 3px;text-transform:uppercase;font-weight:bold">{_esc(row)}</td></tr>')
            continue
        label, value, *rest = row
        status = rest[0] if rest else ''
        sc = ('#2ecc71' if any(x in status for x in ('✅','OK','ONLINE','ACTIVE'))
              else '#f39c12' if any(x in status for x in ('⚠','WARN','LOW','PENDING'))
              else '#e74c3c' if any(x in status for x in ('❌','FAIL','ERROR'))
              else '#666')
        inner += (f'<tr>'
                  f'<td style="color:#555;font-size:10px;letter-spacing:1.5px;text-transform:uppercase;'
                  f'padding:4px 14px 4px 0;white-space:nowrap;vertical-align:top">{_esc(label)}</td>'
                  f'<td style="color:#ddd;font-size:12px;padding:4px 8px 4px 0;'
                  f'font-family:Courier New,monospace;word-break:break-word">{_esc(value)}</td>'
                  f'<td style="color:{sc};font-size:11px;padding:4px 0;white-space:nowrap;'
                  f'vertical-align:top">{_esc(status)}</td></tr>')
    note_html = (f'<div style="color:#444;font-size:10px;margin-top:10px;'
                 f'border-top:1px solid #1a1a1a;padding-top:8px">{_esc(note)}</div>' if note else '')
    return (f'<div style="background:#060606;border:1.5px solid {accent};border-radius:9px;'
            f'padding:18px 22px;margin:10px 0;font-family:Courier New,monospace;'
            f'max-width:{width};box-shadow:0 0 18px {accent}1a">'
            f'<div style="color:{accent};font-size:13px;font-weight:bold;letter-spacing:3px;'
            f'border-bottom:1px solid #1a1a1a;padding-bottom:9px;margin-bottom:12px">'
            f'⬡ {_esc(title)}</div>'
            f'<table style="border-collapse:collapse;width:100%">{inner}</table>'
            f'{note_html}</div>')

def _banner(l1, l2='', l3=''):
    sub = ''
    if l2: sub += f'<div style="color:#f39c12;font-size:10px;letter-spacing:3px;margin-top:4px">{_esc(l2)}</div>'
    if l3: sub += f'<div style="color:#555;font-size:10px;letter-spacing:2px;margin-top:3px">{_esc(l3)}</div>'
    return (f'<div style="background:linear-gradient(135deg,#070707,#1b0303);'
            f'border:2px solid #c0392b;border-radius:12px;padding:20px 26px;margin:10px 0;'
            f'font-family:Courier New,monospace;box-shadow:0 0 28px #c0392b33">'
            f'<div style="color:#e74c3c;font-size:22px;font-weight:bold;letter-spacing:5px">⬡ {_esc(l1)}</div>'
            f'{sub}</div>')

def _alert(msg, level='info'):
    c = {'info':'#3498db','success':'#2ecc71','warn':'#f39c12','error':'#e74c3c'}.get(level,'#888')
    e = {'info':'ℹ️','success':'✅','warn':'⚠️','error':'❌'}.get(level,'•')
    return (f'<div style="background:{c}14;border-left:3px solid {c};padding:9px 14px;'
            f'margin:6px 0;font-family:Courier New,monospace;font-size:12px;'
            f'color:#ccc;border-radius:0 6px 6px 0;max-width:700px">{e} {_esc(msg)}</div>')

def _pbar(pct, label='', color='#e74c3c'):
    p = max(0, min(100, pct))
    lbl = (f'<div style="color:#666;font-size:10px;letter-spacing:1px;margin-bottom:3px;'
           f'text-transform:uppercase">{_esc(label)}</div>') if label else ''
    return (f'{lbl}<div style="background:#111;border-radius:3px;height:5px;max-width:700px">'
            f'<div style="background:linear-gradient(90deg,{color}99,{color});'
            f'width:{p}%;height:100%;border-radius:3px"></div></div>'
            f'<div style="color:{color};font-size:9px;text-align:right;max-width:700px;margin-top:1px">{p:.1f}%</div>')

# ── Diagnostics ──────────────────────────────────────────────────
display(HTML(_banner('J.A.R.V.I.S.', 'VoxCPM2 HINDI/HINGLISH TTS v4 — READLYTE', 'CELL 1 / 7  BOOT')))

rows = []
gpu_ok = False
rows.append('RUNTIME')
rows.append(('Python', sys.version.split()[0], 'ACTIVE'))
rows.append(('Platform', f'{platform.system()} {platform.release()}', ''))
rows.append(None)
rows.append('GPU STATUS')
try:
    raw = subprocess.check_output(
        ['nvidia-smi','--query-gpu=name,memory.total,memory.free,driver_version','--format=csv,noheader'],
        encoding='utf-8').strip().split(',')
    gn, vt, vf, dr = [x.strip() for x in raw[:4]]
    vmb = int(''.join(c for c in vt if c.isdigit()))
    gpu_ok = vmb >= 7000
    rows += [('Device', gn, 'ONLINE'),
             ('VRAM Total', vt, '✅ OK' if gpu_ok else '⚠ LOW (<7 GB)'),
             ('VRAM Free', vf, ''), ('Driver', dr, '')]
except Exception as ex:
    rows.append(('GPU', f'Not detected — {ex}', '❌ OFFLINE'))

rows.append(None)
rows.append('PYTORCH / CUDA')
try:
    import torch
    cuda = torch.cuda.is_available()
    rows += [('PyTorch', torch.__version__, ''),
             ('CUDA', torch.version.cuda if cuda else 'N/A', '✅ OK' if cuda else '❌ NO CUDA'),
             ('GPU', torch.cuda.get_device_name(0) if cuda else '--', '')]
except ImportError:
    rows.append(('PyTorch', 'Not installed — Cell 2 will install it', '⚠ PENDING'))

rows.append(None)
rows.append('STORAGE')
try:
    dk = subprocess.check_output(['df','-h','/'], encoding='utf-8').strip().split('\n')[-1].split()
    used_pct = int(dk[4].replace('%',''))
    rows.append(('Disk (/)', f'{dk[3]} free / {dk[1]} total ({dk[4]} used)',
                 '✅ OK' if used_pct < 85 else '⚠ LOW SPACE'))
except Exception:
    rows.append(('Disk', 'Unable to read', ''))

display(HTML(_panel('SYSTEM DIAGNOSTICS', rows,
    note='VoxCPM2 requires >= 8 GB VRAM and ~15 GB disk (first run model download).')))
if not gpu_ok:
    display(HTML(_alert('T4 GPU REQUIRED — Runtime > Change Runtime Type > T4 GPU > Save', 'warn')))
else:
    display(HTML(_alert('All systems nominal. Proceed to Cell 2.', 'success')))

In [ ]:
# ================================================================
# CELL 2 — INSTALL DEPENDENCIES
# ================================================================
import subprocess, sys, time
from IPython.display import display, HTML

display(HTML(_banner('INSTALLING DEPENDENCIES', 'VoxCPM2 + AUDIO STACK', 'CELL 2 / 7')))

PACKAGES = [
    ('voxcpm',          'VoxCPM2 core TTS library',         True),
    ('transformers',    'HuggingFace model loader',        False),
    ('accelerate',      'Multi-GPU / CPU offload helper',  False),
    ('huggingface_hub', 'HF Hub download utility',         False),
    ('soundfile',       'Audio file read / write',         False),
    ('scipy',           'Signal processing',               False),
    ('librosa',         'Audio analysis utilities',        False),
    ('numpy',           'Numerical arrays',                False),
    ('tqdm',            'Progress bars',                   False),
    ('ipywidgets',      'Colab UI widgets',                False),
]

rows   = []
all_ok = True
for i, (pkg, desc, upgrade) in enumerate(PACKAGES, 1):
    t0     = time.time()
    flags  = ['--upgrade'] if upgrade else []
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'] + flags,
        capture_output=True, text=True)
    elapsed = time.time() - t0
    ok = result.returncode == 0
    if not ok: all_ok = False
    rows.append((f'[{i:02d}/{len(PACKAGES):02d}] {pkg}',
                 f'{desc}  ({elapsed:.1f}s)', '✅ OK' if ok else '❌ FAILED'))

display(HTML(_panel('PACKAGE INSTALLATION', rows,
    note='If voxcpm fails: !pip install git+https://github.com/openbmb/voxcpm.git')))

vrows = []
for mod in ('voxcpm', 'soundfile', 'numpy', 'torch', 'scipy'):
    try:
        m = __import__(mod)
        vrows.append((mod, f"v{getattr(m,'__version__','ok')}", '✅ IMPORTED'))
    except ImportError as e:
        vrows.append((mod, str(e), '❌ FAILED'))
        all_ok = False

display(HTML(_panel('IMPORT VERIFICATION', vrows)))
display(HTML(_alert('All packages installed. Proceed to Cell 3.', 'success') if all_ok
             else _alert('Some imports failed — check errors above.', 'error')))

In [ ]:
# ================================================================
# CELL 3 — UPLOAD TEXT + PREPROCESSING
# ================================================================
#
# ENABLE_NORMALIZATION:
#   True  = Layer 2 ON  — converts curly quotes, en-dash, ellipsis,
#            fixes spacing around punctuation. Recommended default.
#   False = Layer 2 OFF — preserves raw punctuation exactly as written.
#           Use when your text is already editor-clean.
#
# @title Text Preprocessing Settings
ENABLE_NORMALIZATION = True  # @param {type:"boolean"}

# ================================================================
from google.colab import files
from IPython.display import display, HTML
import re, unicodedata

display(HTML(_banner('UPLOAD TEXT FILE',
                     'HINDI  HINGLISH  DEVANAGARI / LATIN / MIXED',
                     'CELL 3 / 7')))
display(HTML(_alert(
    'Upload a UTF-8 .txt file. Paragraphs starting with (style instruction) will control '
    'the TTS voice automatically — no manual tagging needed.', 'info')))

# ── LAYER 1: always-on safe structural cleanup ────────────────────
def _clean_text(text):
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    # Remove purely decorative separator lines (=====, -----)
    text = re.sub(r'(?m)^[ \t]*[=\-]{5,}[ \t]*$', '', text)
    text = unicodedata.normalize('NFC', text)
    # Zero-width / invisible chars
    text = re.sub(r'[\u00ad\u200b\u200c\u200d\u2060\ufeff]', '', text)
    text = re.sub(r'[ \t]+', ' ', text)           # collapse spaces/tabs
    text = re.sub(r'\n{3,}', '\n\n', text)        # max 2 blank lines
    text = '\n'.join(line.rstrip() for line in text.split('\n'))
    return text.strip()

# ── LAYER 2: optional typographic normalization ───────────────────
def _normalize_text(text):
    for src, dst in [
        ('\u2018',"'"),('\u2019',"'"),('\u201a',"'"),('\u201b',"'"),
        ('\u201c','"'),('\u201d','"'),('\u201e','"'),('\u201f','"'),
        ('\u2039',"'"),('\u203a',"'"),('\u00ab','"'),('\u00bb','"'),
    ]:
        text = text.replace(src, dst)
    text = text.replace('\u2013', '-')             # en-dash → hyphen
    text = text.replace('\u2026', '...')           # ellipsis char
    text = re.sub(r'-{3,}', '\u2014', text)        # 3+ dashes → em-dash
    text = re.sub(r' +([।.!?,;:])', r'\1', text)  # space before punct
    text = re.sub(r'([।.!?])([^\s\n।.!?])', r'\1 \2', text)
    text = re.sub(r'([।!?]){3,}', r'\1\1', text)
    text = re.sub(r'\.{4,}', '...', text)
    text = re.sub(r'\u202f', ' ', text)            # narrow no-break space
    return text

def preprocess_for_tts(text, normalize=True):
    text = _clean_text(text)
    if normalize:
        text = _normalize_text(text)
    return text

# ── Upload ────────────────────────────────────────────────────────
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded. Re-run this cell and upload a .txt file.')

fname = list(uploaded.keys())[0]
if not fname.lower().endswith('.txt'):
    raise ValueError(f'Expected a .txt file, got: {fname}')

raw = uploaded[fname]
try:
    raw_text = raw.decode('utf-8')
except UnicodeDecodeError:
    raw_text = raw.decode('utf-8', errors='replace')
    display(HTML(_alert('Some bytes could not be decoded — replaced with U+FFFD. '
                        'Re-save as UTF-8 for cleanest results.', 'warn')))

INPUT_TEXT_RAW   = raw_text.strip()
INPUT_TEXT_CLEAN = _clean_text(INPUT_TEXT_RAW)
INPUT_TEXT       = preprocess_for_tts(INPUT_TEXT_RAW, normalize=ENABLE_NORMALIZATION)

# ── Stats ─────────────────────────────────────────────────────────
def _count_diffs(a, b):
    from difflib import SequenceMatcher
    return sum(max(j1-i1, j2-i2)
               for tag, i1, i2, j1, j2 in SequenceMatcher(None, a, b, autojunk=False).get_opcodes()
               if tag != 'equal')

char_count    = len(INPUT_TEXT)
word_count    = len(INPUT_TEXT.split())
para_count    = len([p for p in INPUT_TEXT.split('\n\n') if p.strip()])
est_audio     = word_count / 130
changes_clean = _count_diffs(INPUT_TEXT_RAW, INPUT_TEXT_CLEAN)
changes_norm  = _count_diffs(INPUT_TEXT_CLEAN, INPUT_TEXT) if ENABLE_NORMALIZATION else 0

# Count style-cued paragraphs
_CUE_DETECT = re.compile(r'^\s*\(([^)\n]{3,300})\)')
cued_paras = sum(1 for p in INPUT_TEXT.split('\n\n') if p.strip() and _CUE_DETECT.match(p.strip()))

rows = [
    ('Filename',          fname,                                      ''),
    ('Characters',        f'{char_count:,}',                         ''),
    ('Words',             f'{word_count:,}',                         ''),
    ('Paragraphs',        f'{para_count:,}',                         ''),
    ('Style-cued paras',  f'{cued_paras} / {para_count}',
     '✅ OK' if cued_paras > 0 else '⚠ NONE (no tone instructions found)'),
    ('Est. Duration',     f'~{est_audio:.1f} min @ 130 wpm',         ''),
    None,
    'PREPROCESSING PIPELINE',
    ('Layer 1  Clean',
     f'{changes_clean} fix(es) — NFC, ZW-chars, whitespace', '✅ DONE'),
    ('Layer 2  Normalize',
     'ENABLED' if ENABLE_NORMALIZATION else 'DISABLED (raw text kept)',
     '✅ ON' if ENABLE_NORMALIZATION else '⚠ OFF'),
    ('  Norm changes',
     f'{changes_norm} typographic fix(es)' if ENABLE_NORMALIZATION else 'skipped', ''),
]
display(HTML(_panel('TEXT FILE STATS', rows,
    note='Style instructions in (parentheses) at the start of a paragraph control voice tone per-paragraph.')))

preview_html = ('<div style="background:#0a0a0a;border:1px solid #2a2a2a;border-radius:6px;'
                'padding:12px 16px;margin:8px 0;font-family:Courier New,monospace;font-size:12px;'
                'color:#aaa;max-width:700px;white-space:pre-wrap;max-height:220px;overflow-y:auto;'
                'line-height:1.6">' + _esc(INPUT_TEXT[:700])
                + ('<span style="color:#555">...</span>' if len(INPUT_TEXT) > 700 else '')
                + '</div>')
display(HTML('<div style="color:#666;font-size:10px;letter-spacing:2px;text-transform:uppercase;'
             'margin-top:12px">PREVIEW — FIRST 700 CHARS</div>'))
display(HTML(preview_html))
display(HTML(_alert('Text loaded and pre-processed. Proceed to Cell 4.', 'success')))

In [ ]:
# ================================================================
# CELL 4 — REFERENCE AUDIO (OPTIONAL — SKIP FOR DEFAULT VOICE)
# ================================================================
from google.colab import files
from IPython.display import display, HTML
import os, subprocess
import soundfile as sf
import numpy as np

display(HTML(_banner('REFERENCE AUDIO', 'VOICE CLONING CONFIG', 'CELL 4 / 7')))

display(HTML(_panel('CLONING MODES', [
    ('Default TTS',    'No reference audio. VoxCPM2 built-in voice.', ''),
    ('Voice Clone',    'Reference WAV/MP3/FLAC — style cloning.', ''),
    ('Ultimate Clone', 'Reference audio + exact transcript — highest fidelity.', ''),
], note='Upload a 10-20 s clean, noise-free single-speaker clip for best cloning results.')))

print('\n>>> UPLOAD your reference audio file (or press Cancel to skip):')
ref_uploaded = files.upload()

REFERENCE_WAV_PATH = None
PROMPT_WAV_PATH    = None
PROMPT_TEXT        = None

if not ref_uploaded:
    display(HTML(_alert('No reference audio — Default TTS mode active.', 'info')))
else:
    ref_fname = list(ref_uploaded.keys())[0]
    ref_ext   = os.path.splitext(ref_fname)[1].lower()
    if ref_ext not in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'):
        raise ValueError(f'Unsupported format: {ref_ext}. Use WAV / MP3 / FLAC / OGG / M4A.')

    ref_save = f'/content/reference_audio{ref_ext}'
    with open(ref_save, 'wb') as fh:
        fh.write(ref_uploaded[ref_fname])

    if ref_ext != '.wav':
        wav_out = '/content/reference_audio.wav'
        r = subprocess.run(
            ['ffmpeg', '-y', '-i', ref_save, '-ar', '16000', '-ac', '1', wav_out],
            capture_output=True)
        if r.returncode == 0:
            ref_save = wav_out
            display(HTML(_alert(f'Converted {ref_ext} → WAV 16 kHz mono.', 'success')))
        else:
            display(HTML(_alert('ffmpeg conversion failed — using original file.', 'warn')))

    try:
        data, sr = sf.read(ref_save, always_2d=False)
        dur      = len(data) / sr
        peak_val = float(np.abs(data).max())
        qual     = ('✅ OK (8-30 s ideal)' if 8 <= dur <= 30
                    else '⚠ VERY SHORT (<8 s)' if dur < 8
                    else '⚠ LONG (>30 s) — trim to 10-20 s')
        display(HTML(_panel('REFERENCE AUDIO ANALYSIS', [
            ('File',        ref_fname,                                                 ''),
            ('Sample Rate', f'{sr} Hz',                                               ''),
            ('Duration',    f'{dur:.2f} s',                                           qual),
            ('Channels',    'Stereo' if data.ndim == 2 else 'Mono',                  ''),
            ('Peak Level',  f'{peak_val:.4f}',
             '✅ OK' if peak_val > 0.05 else '⚠ VERY QUIET — normalise before use'),
        ])))
    except Exception as e:
        display(HTML(_alert(f'Could not read audio stats: {e}', 'warn')))

    REFERENCE_WAV_PATH = ref_save
    PROMPT_WAV_PATH    = ref_save

    print('\n>>> Paste the EXACT transcript of your reference audio clip.')
    print('    (Leave blank to use Voice Clone mode without a transcript)')
    prompt_input = input('Reference transcript: ').strip()
    if prompt_input:
        PROMPT_TEXT = prompt_input
        display(HTML(_alert('Ultimate Clone mode active (reference + transcript).', 'success')))
    else:
        display(HTML(_alert('Voice Clone mode active (reference only, no transcript).', 'success')))

display(HTML(_alert('Voice config set. Proceed to Cell 5.', 'success')))

In [ ]:
# ================================================================
# CELL 5 — TTS CONFIGURATION
# ================================================================
#
# ┌──────────────────────────────────────────────────────────────┐
# │  ══════════════  USER CONFIG  ════════════════════════════   │
# │  Edit ONLY the values in this block.                        │
# │  Re-run this cell after any change.                         │
# └──────────────────────────────────────────────────────────────┘

# ── Output file ──────────────────────────────────────────────────
OUTPUT_FILENAME  = 'jarvis_audiobook_output'   # filename, no extension
OUTPUT_PATH      = f'/content/{OUTPUT_FILENAME}.wav'

# ── Chunk size (characters per TTS call) ─────────────────────────
# Audiobooks: 350-420 chars is ideal — longer = more natural prosody.
# VoxCPM2 safe context limit: <= 480 chars. Do NOT exceed 480.
CHUNK_SIZE       = 380   # recommended: 320-420

# ── Minimum chunk size ───────────────────────────────────────────
# Chunks shorter than this are merged with the previous chunk.
MIN_CHUNK_SIZE   = 80    # chars; recommended: 60-100

# ── Inference quality ─────────────────────────────────────────────
# 10 = fast draft,  20 = balanced,  32 = best quality (commercial)
INFERENCE_TIMESTEPS = 32

# ── CFG strength (style adherence) ────────────────────────────────
# 1.5 = loose/creative,  2.0 = balanced (recommended),  3.0 = strict
CFG_VALUE        = 2.0

# ── Silence durations between chunks (seconds) ───────────────────
SILENCE_CHAPTER   = 2.00   # after detected chapter headings
SILENCE_PARAGRAPH = 0.55   # after paragraph boundary
SILENCE_SENTENCE  = 0.20   # after sentence boundary (mid-paragraph)
SILENCE_CLAUSE    = 0.08   # after clause-level split (comma / dash)

# ── Crossfade at chunk tail (milliseconds) ────────────────────────
# Eliminates click artifacts at stitch points. 10-20 ms recommended.
CROSSFADE_MS     = 15

# ── Resume / checkpoint ───────────────────────────────────────────
# True = cache chunks as .npy; only failed chunks re-generated on re-run.
RESUME_CHUNKS    = True
CHUNKS_DIR       = '/content/chunks'

# ─────────────────────────────────────────────────────────────────
#  END OF USER CONFIG
# ─────────────────────────────────────────────────────────────────
from IPython.display import display, HTML

if 'ENABLE_NORMALIZATION' not in dir():
    ENABLE_NORMALIZATION = True

CHUNK_SIZE          = max(150, min(480, int(CHUNK_SIZE)))
MIN_CHUNK_SIZE      = max(30,  min(200, int(MIN_CHUNK_SIZE)))
INFERENCE_TIMESTEPS = int(INFERENCE_TIMESTEPS) if INFERENCE_TIMESTEPS in (10, 20, 32) else 20
CFG_VALUE           = float(CFG_VALUE) if 0.5 <= float(CFG_VALUE) <= 5.0 else 2.0
CROSSFADE_MS        = max(0, min(50, int(CROSSFADE_MS)))
SILENCE_CHAPTER     = max(0.0, float(SILENCE_CHAPTER))
SILENCE_PARAGRAPH   = max(0.0, float(SILENCE_PARAGRAPH))
SILENCE_SENTENCE    = max(0.0, float(SILENCE_SENTENCE))
SILENCE_CLAUSE      = max(0.0, float(SILENCE_CLAUSE))

rows = [
    ('Output File',      f'{OUTPUT_FILENAME}.wav', ''),
    None, 'CHUNKING',
    ('Chunk Size',       f'{CHUNK_SIZE} chars', ''),
    ('Min Chunk Size',   f'{MIN_CHUNK_SIZE} chars', ''),
    None, 'INFERENCE',
    ('Timesteps',        str(INFERENCE_TIMESTEPS),
     'FAST' if INFERENCE_TIMESTEPS == 10 else 'BALANCED' if INFERENCE_TIMESTEPS == 20 else 'BEST'),
    ('CFG Strength',     str(CFG_VALUE), ''),
    ('Voice Control',    'Per-paragraph (instruction) tags in text — full text control', '✅ ACTIVE'),
    None, 'SILENCE GAPS',
    ('Chapter Break',    f'{SILENCE_CHAPTER*1000:.0f} ms', ''),
    ('Paragraph Break',  f'{SILENCE_PARAGRAPH*1000:.0f} ms', ''),
    ('Sentence Break',   f'{SILENCE_SENTENCE*1000:.0f} ms', ''),
    ('Clause Break',     f'{SILENCE_CLAUSE*1000:.0f} ms', ''),
    None, 'STITCHING + RESUME',
    ('Crossfade',        f'{CROSSFADE_MS} ms fade-out per chunk',
     'ACTIVE' if CROSSFADE_MS > 0 else 'OFF'),
    ('Resume Chunks',    'ENABLED' if RESUME_CHUNKS else 'DISABLED',
     '✅ ON' if RESUME_CHUNKS else '⚠ OFF'),
    None, 'PREPROCESSING',
    ('Normalization',
     'ENABLED (Layer 2 ON)' if ENABLE_NORMALIZATION else 'DISABLED (raw text)',
     '✅ ON' if ENABLE_NORMALIZATION else '⚠ OFF'),
]
display(HTML(_banner('TTS CONFIGURATION', 'PARAMETERS LOCKED IN', 'CELL 5 / 7')))
display(HTML(_panel('ACTIVE PARAMETERS', rows,
    note='Edit the USER CONFIG block above and re-run to update.')))
display(HTML(_alert('Parameters confirmed. Proceed to Cell 6 to generate TTS.', 'success')))

In [ ]:
# ================================================================
# CELL 6 — LOAD MODEL + GENERATE TTS
# ================================================================
import re, time, os, gc
import unicodedata
import numpy as np
import soundfile as sf
import torch
from IPython.display import display, HTML, clear_output
from voxcpm import VoxCPM

assert 'INPUT_TEXT' in globals() and INPUT_TEXT, 'Run Cell 3 first — INPUT_TEXT not defined.'
assert 'CHUNK_SIZE' in globals(), 'Run Cell 5 first — TTS parameters not set.'
assert 'REFERENCE_WAV_PATH' in globals(), 'Run Cell 4 first — voice clone config not set.'

display(HTML(_banner('TTS GENERATION ENGINE', 'VoxCPM2  AUDIOBOOK MODE  v4', 'CELL 6 / 7')))


# ════════════════════════════════════════════════════════════════
#  AUDIOBOOK CHUNKER v4 — FIXED STYLE CUE HANDLING
#
#  VoxCPM2 natively understands the format: (instruction)text
#  The parenthetical instruction MUST be at the very start of the
#  text string passed to model.generate(). This chunker:
#    1. Strips the leading (instruction) from each paragraph.
#    2. Re-attaches it to EVERY chunk from that paragraph.
#    3. Passes the combined string directly to model.generate().
#  No STYLE_PREFIX is used — voice is 100% text-controlled.
# ════════════════════════════════════════════════════════════════

_TERMINAL_CHARS = set('।.!?')
_SENT_RE        = re.compile(r'(?<=[।.!?])\s+')
_CLAUSE_RE      = re.compile(r'(?<=[,;\u2014])\s+')
_CHAPTER_RE     = re.compile(
    r'^(?:chapter\s*[\divxlc]+\b'
    r'|chapter\s+\w+\b'
    r'|\u0905\u0927\u094d\u092f\u093e\u092f\s*\d+'
    r'|\u092d\u093e\u0917\s*\d+'
    r'|part\s*[\divxlc]+\b'
    r'|prologue|epilogue|preface|foreword|afterword'
    r'|\u092a\u094d\u0930\u0938\u094d\u0924\u093e\u0935\u0928\u093e'
    r'|\u0909\u092a\u0938\u0902\u0939\u093e\u0930'
    r'|[=\-]{5,}'
    r'|#{1,3}\s+)',
    re.IGNORECASE)

# ── FIXED: Style cue regex ────────────────────────────────────────
# Changes from v3:
#   • Min inner length: 5 → 3  (catches short cues like "slow")
#   • Removed the sentence-punctuation rejection test — a cue like
#     "warm, measured pace" has a comma but is clearly a style tag,
#     not a dialogue fragment. We now rely purely on position (start
#     of paragraph) and balanced parentheses to identify cues.
#   • Increased max inner length to 300 chars.
_STYLE_CUE_RE = re.compile(r'^\(([^)\n]{3,300})\)\s*')


def _extract_style_cue(para_text):
    """Extract a leading (style instruction) from a paragraph.

    Returns (cue_string, remaining_content) where cue_string is the
    full '(instruction)' token, or (None, para_text) if none found.

    The cue is then prepended to EVERY chunk produced from the
    paragraph so VoxCPM2 receives consistent voice guidance on each
    model.generate() call.
    """
    m = _STYLE_CUE_RE.match(para_text.strip())
    if m:
        cue  = '(' + m.group(1).strip() + ')'
        rest = para_text.strip()[m.end():].strip()
        return cue, rest
    return None, para_text


# Separator chunk guard (e.g. lines of ===== after punct was appended)
_SEPARATOR_CHUNK_RE = re.compile(r'^[=\-#*]{5,}[.。]?\s*$')


def _ensure_terminal_punct(text):
    t = text.rstrip()
    if not t or t[-1] in _TERMINAL_CHARS:
        return t
    if re.search(r'[\u0900-\u097F]\s*$', t):
        return t + '\u0964'   # Devanagari danda
    return t + '.'


def _hard_split_words(text, n):
    parts, buf = [], ''
    for word in text.split():
        test = (buf + ' ' + word).strip()
        if len(test) <= n:
            buf = test
        else:
            if buf: parts.append(buf)
            buf = word[:n] if len(word) > n else word
    if buf: parts.append(buf)
    return parts


def _merge_chunks(items, max_c, min_c=0):
    chunks, buf = [], ''
    for item in items:
        item = item.strip()
        if not item: continue
        if len(item) > max_c:
            if buf: chunks.append(buf); buf = ''
            chunks.extend(_hard_split_words(item, max_c))
        elif len(buf) + len(item) + 1 <= max_c:
            buf = (buf + ' ' + item).strip()
        else:
            if buf: chunks.append(buf)
            buf = item
    if buf: chunks.append(buf)
    merged = []
    for ch in chunks:
        if merged and len(ch) < min_c and len(merged[-1]) + len(ch) + 1 <= max_c:
            merged[-1] = (merged[-1] + ' ' + ch).strip()
        else:
            merged.append(ch)
    return [c for c in merged if c]


def smart_chunk_audiobook(text, max_chars, min_chars):
    """Split text into (chunk_text, break_type) pairs.

    Each chunk_text is ready to be passed directly to model.generate().
    If the source paragraph had a leading (style cue), it is prepended
    to every chunk derived from that paragraph — ensuring the model
    receives voice guidance on every single inference call.
    """
    paragraphs = [p.strip() for p in re.split(r'\n\n+', text) if p.strip()]
    result     = []
    n_para     = len(paragraphs)

    for p_idx, para in enumerate(paragraphs):
        is_last_para = (p_idx == n_para - 1)

        # Chapter / section heading: one chunk, chapter break after
        if ('\n' not in para.strip()
                and len(para) < 120
                and _CHAPTER_RE.match(para.strip())):
            chunk = _ensure_terminal_punct(para.strip())
            result.append((chunk, 'end' if is_last_para else 'chapter'))
            continue

        # ── Extract leading style cue ─────────────────────────────
        # The cue is removed from the content being chunked, then
        # re-attached to every resulting chunk before it is passed
        # to the model. This is the key fix: VoxCPM2 only honours
        # the (instruction) when it is at position 0 of the text
        # string, so we must not split it away from its content.
        para_cue, para_content = _extract_style_cue(para)

        # Reduce effective chunk size to leave room for the cue header
        cue_overhead  = len(para_cue) + 1 if para_cue else 0
        effective_max = max(80, max_chars - cue_overhead)

        # Sentence-level split on the CONTENT portion (cue removed)
        sentences = [s.strip() for s in _SENT_RE.split(para_content) if s.strip()]

        expanded = []
        for sent in sentences:
            if len(sent) <= effective_max:
                expanded.append(sent)
            else:
                clauses = [c.strip() for c in _CLAUSE_RE.split(sent) if c.strip()]
                if len(clauses) > 1:
                    expanded.extend(_merge_chunks(clauses, effective_max, 0))
                else:
                    expanded.extend(_hard_split_words(sent, effective_max))

        para_chunks = _merge_chunks(expanded, effective_max, min_chars)
        n_pc        = len(para_chunks)

        for c_idx, chunk in enumerate(para_chunks):
            is_last_chunk = (c_idx == n_pc - 1)
            chunk = _ensure_terminal_punct(chunk)
            # ── KEY FIX: re-attach style cue to EVERY chunk ───────
            # Without this, only the first chunk of a long paragraph
            # would carry the voice instruction; subsequent chunks
            # would fall back to the model's default tone.
            if para_cue:
                chunk = para_cue + chunk
            if is_last_chunk and is_last_para:
                btype = 'end'
            elif is_last_chunk:
                btype = 'paragraph'
            else:
                btype = 'sentence'
            result.append((chunk, btype))

    return result


# ════════════════════════════════════════════════════════════════
#  CROSSFADE STITCHER
# ════════════════════════════════════════════════════════════════

def stitch_audio(segments, gap_types, silence_map, sr, fade_ms):
    fade_n = max(0, int(fade_ms * sr / 1000))
    pieces = []
    fade_curve = np.linspace(1.0, 0.0, fade_n, dtype=np.float32) if fade_n > 0 else None
    for i, seg in enumerate(segments):
        seg_arr = np.asarray(seg, dtype=np.float32)
        if seg_arr.size > 0:
            if fade_curve is not None and len(seg_arr) > fade_n * 3:
                seg_arr = seg_arr.copy()
                seg_arr[-fade_n:] *= fade_curve
            pieces.append(seg_arr)
        if i < len(gap_types):
            sil_sec = silence_map.get(gap_types[i], 0.0)
            if sil_sec > 0:
                pieces.append(np.zeros(int(sil_sec * sr), dtype=np.float32))
    return np.concatenate(pieces).astype(np.float32) if pieces else np.array([], dtype=np.float32)


# ════════════════════════════════════════════════════════════════
#  STEP A — CHUNKING
# ════════════════════════════════════════════════════════════════
print('Chunking text ...')
chunk_pairs  = smart_chunk_audiobook(INPUT_TEXT, CHUNK_SIZE, MIN_CHUNK_SIZE)
total_chunks = len(chunk_pairs)
total_chars  = sum(len(c) for c, _ in chunk_pairs)
break_counts = {}
for _, b in chunk_pairs: break_counts[b] = break_counts.get(b, 0) + 1

# Count how many chunks carry a style cue
cue_chunks = sum(1 for c, _ in chunk_pairs if c.startswith('('))

chunk_rows = [
    ('Total Chunks',     str(total_chunks),                                  ''),
    ('Style-cued chunks', f'{cue_chunks} / {total_chunks}',
     '✅ OK' if cue_chunks > 0 else '⚠ NONE — check your (instruction) tags'),
    ('Total Characters', f'{total_chars:,}',                                ''),
    ('Avg Chunk Size',   f'{total_chars // max(total_chunks, 1)} chars',    ''),
    None, 'BREAK DISTRIBUTION',
] + [(f'  {k} breaks', str(v), '') for k, v in sorted(break_counts.items())]
display(HTML(_panel('CHUNKING SUMMARY', chunk_rows, accent='#f39c12')))

preview_rows = []
for i, (c, bt) in enumerate(chunk_pairs[:6], 1):
    preview_rows.append((f'Chunk {i:03d}  [{bt}]',
                         c[:90] + ('...' if len(c) > 90 else ''), ''))
if total_chunks > 6:
    preview_rows.append(('...', f'and {total_chunks - 6} more chunks', ''))
display(HTML(_panel('CHUNK PREVIEW (first 6)', preview_rows, accent='#f39c12')))


# ════════════════════════════════════════════════════════════════
#  STEP B — LOAD MODEL
# ════════════════════════════════════════════════════════════════
print('\nLoading VoxCPM2 model (~8 GB download on first run) ...')
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

t_load = time.time()
model  = VoxCPM.from_pretrained('openbmb/VoxCPM2', load_denoiser=True)
load_time = time.time() - t_load

SAMPLE_RATE = None
for _attr in ('tts_model.sample_rate', 'sample_rate'):
    try:
        obj = model
        for part in _attr.split('.'): obj = getattr(obj, part)
        SAMPLE_RATE = int(obj); break
    except AttributeError:
        pass
if SAMPLE_RATE is None:
    SAMPLE_RATE = 48_000
    display(HTML(_alert('sample_rate not found — defaulting to 48 000 Hz.', 'warn')))

model_rows = [
    ('Model',       'openbmb/VoxCPM2',    ''),
    ('Load Time',   f'{load_time:.1f} s', '✅ OK'),
    ('Sample Rate', f'{SAMPLE_RATE:,} Hz',''),
]
if torch.cuda.is_available():
    used_gb  = torch.cuda.memory_allocated() / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    model_rows.append(('VRAM Used', f'{used_gb:.2f} GB / {total_gb:.2f} GB',
                        '✅ OK' if used_gb < total_gb * 0.85 else '⚠ HIGH'))
display(HTML(_panel('MODEL STATUS', model_rows, accent='#2ecc71')))


# ════════════════════════════════════════════════════════════════
#  STEP C — GENERATE CHUNK BY CHUNK
# ════════════════════════════════════════════════════════════════
if RESUME_CHUNKS:
    os.makedirs(CHUNKS_DIR, exist_ok=True)

SILENCE_MAP = {
    'chapter':   SILENCE_CHAPTER,
    'paragraph': SILENCE_PARAGRAPH,
    'sentence':  SILENCE_SENTENCE,
    'clause':    SILENCE_CLAUSE,
    'end':       0.0,
}

mode_label = ('Ultimate Clone' if (REFERENCE_WAV_PATH and PROMPT_TEXT)
               else 'Voice Clone' if REFERENCE_WAV_PATH
               else 'Default TTS')

all_audio    = []
chunk_breaks = []
chunk_times  = []
failed_idx   = []
resumed_idx  = []
gen_start    = time.time()


def _render_progress(done, total, ctimes, failed, resumed, eta_str, current, btype):
    pct   = (done / total) * 100 if total else 0
    bar_f = int(pct / 100 * 28)
    bar   = chr(0x2588) * bar_f + chr(0x2591) * (28 - bar_f)
    active_t = [t for i, t in enumerate(ctimes) if (i + 1) not in set(resumed) and t > 0]
    avg  = sum(active_t) / len(active_t) if active_t else 0
    rows = [
        ('Progress',      f'[{bar}]  {pct:.1f}%  ({done}/{total})', ''),
        ('Current Chunk', f"{current[:70]}{'...' if len(current) > 70 else ''}", ''),
        ('Break After',   btype,                                     ''),
        ('Avg / Chunk',   f'{avg:.1f} s' if avg else '--',           ''),
        ('ETA',           eta_str,                                   ''),
        ('Resumed',       str(len(resumed)) if resumed else 'None',  ''),
        ('Failed',        str(len(failed))  if failed  else 'None',
         '' if not failed else '⚠ CHECK'),
    ]
    return (_panel('TTS GENERATION PROGRESS', rows, accent='#e74c3c', width='700px')
            + _pbar(pct, color='#e74c3c'))


for idx, (chunk, btype) in enumerate(chunk_pairs, 1):
    chunk_path = (os.path.join(CHUNKS_DIR, f'chunk_{idx:04d}.npy')
                  if RESUME_CHUNKS else None)

    # Pure separator chunk — no audio, only the gap silence
    if _SEPARATOR_CHUNK_RE.match(chunk.strip()):
        all_audio.append(np.array([], dtype=np.float32))
        chunk_breaks.append(btype)
        chunk_times.append(0.0)
        continue

    # Resume: load from cache if available and valid
    if RESUME_CHUNKS and chunk_path and os.path.exists(chunk_path):
        try:
            wav = np.load(chunk_path)
            if wav.size > 0:
                all_audio.append(wav.astype(np.float32))
                chunk_breaks.append(btype)
                chunk_times.append(0.0)
                resumed_idx.append(idx)
                continue
        except Exception:
            pass  # corrupt cache → regenerate

    t_chunk = time.time()

    real_times = [t for i, t in enumerate(chunk_times)
                  if (i + 1) not in set(resumed_idx) and t > 0]
    if real_times:
        avg_t   = sum(real_times) / len(real_times)
        eta_sec = avg_t * (total_chunks - idx + 1)
        eta_str = f'{eta_sec/60:.1f} min' if eta_sec > 60 else f'{eta_sec:.0f} s'
    else:
        eta_str = 'calculating...'

    clear_output(wait=True)
    display(HTML(_banner('TTS GENERATION', f'Mode: {mode_label}', 'CELL 6 / 7')))
    display(HTML(_render_progress(
        idx - 1, total_chunks, chunk_times,
        failed_idx, resumed_idx, eta_str, chunk, btype)))

    # ── PASS CHUNK DIRECTLY TO MODEL ─────────────────────────────
    # chunk already contains the (style cue) at position 0 if the
    # source paragraph had one. No STYLE_PREFIX is added — voice is
    # 100% controlled by the per-paragraph tags in the source text.
    gen_kwargs = dict(
        text=chunk,
        cfg_value=CFG_VALUE,
        inference_timesteps=INFERENCE_TIMESTEPS,
    )
    if REFERENCE_WAV_PATH:
        gen_kwargs['reference_wav_path'] = REFERENCE_WAV_PATH
    if PROMPT_WAV_PATH and PROMPT_TEXT:
        gen_kwargs['prompt_wav_path'] = PROMPT_WAV_PATH
        gen_kwargs['prompt_text']     = PROMPT_TEXT

    wav = None
    for attempt in range(3):
        try:
            wav = model.generate(**gen_kwargs)
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(4)
            if attempt == 2:
                display(HTML(_alert(
                    f'Chunk {idx}: VRAM OOM after 3 attempts — skipped. '
                    'Re-run with RESUME_CHUNKS=True to retry.', 'error')))
        except Exception as e:
            if attempt < 2:
                time.sleep(2); continue
            display(HTML(_alert(f'Chunk {idx} failed after 3 attempts: {e}', 'error')))

    if wav is None:
        failed_idx.append(idx)
        continue

    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim == 2:
        wav = wav.mean(axis=0)   # stereo → mono

    if wav.size == 0:
        failed_idx.append(idx)
        continue

    wav = np.clip(wav, -1.0, 1.0)

    if RESUME_CHUNKS and chunk_path:
        try:
            np.save(chunk_path, wav)
        except Exception as e:
            display(HTML(_alert(f'Cache write failed chunk {idx}: {e}', 'warn')))

    all_audio.append(wav)
    chunk_breaks.append(btype)
    chunk_times.append(time.time() - t_chunk)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ════════════════════════════════════════════════════════════════
#  STEP D — STITCH + SAVE
# ════════════════════════════════════════════════════════════════
clear_output(wait=True)
display(HTML(_banner('GENERATION COMPLETE', 'STITCHING AND SAVING AUDIO', 'CELL 6 / 7')))

if not all_audio:
    raise RuntimeError('No audio segments generated. Check errors printed above.')

print(f'Stitching {len(all_audio)} segments (crossfade={CROSSFADE_MS} ms) ...')
final_audio = stitch_audio(all_audio, chunk_breaks, SILENCE_MAP, SAMPLE_RATE, CROSSFADE_MS)

# Peak-normalise to -1 dBFS
peak = float(np.abs(final_audio).max())
if peak > 0:
    final_audio = (final_audio / peak * (10 ** (-1.0 / 20))).astype(np.float32)

sf.write(OUTPUT_PATH, final_audio, SAMPLE_RATE, subtype='PCM_16')

total_time = time.time() - gen_start
audio_dur  = len(final_audio) / SAMPLE_RATE
file_mb    = os.path.getsize(OUTPUT_PATH) / 1e6
real_times = [t for i, t in enumerate(chunk_times)
              if (i + 1) not in set(resumed_idx) and t > 0]
avg_ct     = sum(real_times) / len(real_times) if real_times else 0
rtf        = total_time / audio_dur if audio_dur > 0 else 0

display(HTML(_panel('FINAL REPORT', [
    ('Chunks OK / Total',  f'{len(all_audio)} / {total_chunks}',
     '✅ OK' if not failed_idx else f'⚠ {len(failed_idx)} failed'),
    ('Resumed from Cache', str(len(resumed_idx)) if resumed_idx else 'None', ''),
    ('Total Wall Time',    f'{total_time/60:.1f} min  ({total_time:.0f} s)', ''),
    ('Avg Gen / Chunk',    f'{avg_ct:.1f} s' if avg_ct else 'N/A (all resumed)', ''),
    ('Audio Duration',     f'{audio_dur:.1f} s  ({audio_dur/60:.1f} min)', ''),
    ('Real-Time Factor',   f'{rtf:.2f}x', ''),
    ('Sample Rate',        f'{SAMPLE_RATE:,} Hz', ''),
    ('Output File Size',   f'{file_mb:.2f} MB', ''),
    ('Saved To',           OUTPUT_PATH, '✅ WRITTEN'),
], accent='#2ecc71', note='Proceed to Cell 7 to play back and download.')))

if failed_idx:
    display(HTML(_alert(
        f'Failed chunks: {failed_idx} — re-run Cell 6 with RESUME_CHUNKS=True '
        'to regenerate only these.', 'warn')))
display(HTML(_alert('ALL SYSTEMS GREEN  ·  MISSION SUCCESS  ·  Proceed to Cell 7.', 'success')))

In [ ]:
# ================================================================
# CELL 7 — PLAYBACK & DOWNLOAD
# ================================================================
from google.colab import files
from IPython.display import Audio, display, HTML
import soundfile as sf, os, numpy as np

display(HTML(_banner('PLAYBACK & DOWNLOAD', 'JARVIS SESSION FINALIZING', 'CELL 7 / 7')))

if not os.path.exists(OUTPUT_PATH):
    raise FileNotFoundError(f'Output not found: {OUTPUT_PATH}\nRun Cell 6 first.')

data, sr = sf.read(OUTPUT_PATH, always_2d=False)
dur      = len(data) / sr
size_mb  = os.path.getsize(OUTPUT_PATH) / 1e6
peak_val = float(np.abs(data).max())
peak_db  = 20 * np.log10(peak_val) if peak_val > 0 else float('-inf')

display(HTML(_panel('OUTPUT FILE', [
    ('Filename',    f'{OUTPUT_FILENAME}.wav',                    ''),
    ('Duration',    f'{dur:.1f} s  ({dur/60:.1f} min)',         ''),
    ('Sample Rate', f'{sr:,} Hz',                               ''),
    ('File Size',   f'{size_mb:.2f} MB',                       ''),
    ('Peak Level',  f'{peak_db:.2f} dBFS',
     '✅ OK' if -3.0 <= peak_db <= 0.0 else '⚠ CHECK LEVEL'),
], accent='#2ecc71')))

print('\nIN-NOTEBOOK PLAYBACK:')
display(Audio(OUTPUT_PATH, autoplay=False))

print('\nInitiating download ...')
files.download(OUTPUT_PATH)

display(HTML(_alert(f'Download initiated — check your browser for {OUTPUT_FILENAME}.wav', 'success')))